# Permittivity parameters of all PCP-SAFT substances

Results of `fit_solvent_permittivity.py` (procedure of Rueben et al. 2024a with ePCP-SAFT densities) for all substances
of `parameters/epcpsaft/esper2023.json` with DDB permittivity data after the filter of Rueben et al. (2024a) without the pressure limit.

```
export EPCPSAFT_DDB_PERMITTIVITY=<DDB export>
export EPCPSAFT_RUEBEN_REPO=<clone of the repository of Rueben et al. 2024a>   # comparison and n < 5 schemes
python fit_solvent_permittivity.py --write-parameters
```

Outputs are derived from licensed data: execute with `--output-dir results`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv("results/solvent_permittivity.csv")
print(len(r), r.status.str[:70].value_counts().to_dict())

## Schemes and errors

In [ ]:
ok = r[r.status == "ok"].copy()
ok["n_class"] = pd.cut(ok.n, [0, 1, 4, 1e9], labels=["1", "2-4", ">=5"])
stats = ["MAD", "MARD%", "RMSE", "RMSRE%", "bias", "rel_bias%", "max_AD", "max_RD%"]
ok.groupby(["n_class", "scheme"], observed=True)[stats].median().round(3).assign(count=ok.groupby(["n_class", "scheme"], observed=True).size())

Scheme source for fewer than 5 points (Rueben et al. 2024a: DIPPR/ORCA dipole moment) and prescribed schemes:

In [ ]:
ok.scheme_source.value_counts(dropna=False)

## Comparison with the PC-SAFT database of Rueben et al. (2024a)
`PermittivityParameterDatabasePCSAFT.csv` was fitted with the pre-release Esper parameters of her repository; `rueben_MAD_here` evaluates her parameters with the densities used here.

In [ ]:
c = ok[ok.rueben_scheme.notna()]
print("substances in both:", len(c), " same scheme:", (c.scheme == c.rueben_scheme).sum())
pd.DataFrame({"refit": c[["MAD", "MARD%"]].median(), "Rueben parameters, this density": c[["rueben_MAD_here", "rueben_MARD_here"]].median().values})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(np.log10(ok["MARD%"].clip(1e-4)), bins=50)
ax[0].set_xlabel("log10(MARD / %)"); ax[0].set_ylabel("substances")
ax[1].loglog(c["rueben_MARD_here"].clip(1e-4), c["MARD%"].clip(1e-4), ".", ms=3)
ax[1].plot([1e-4, 1e3], [1e-4, 1e3], "k-", lw=0.5)
ax[1].set_xlabel("MARD / %, Rueben parameters"); ax[1].set_ylabel("MARD / %, refit")
plt.tight_layout()

## Solvents of the data specification
Piperazine (110-85-0) is not in Esper et al. (2023) and not in the DDB export.

In [ ]:
target = {"7732-18-5": "water", "7664-41-7": "NH3", "141-43-5": "MEA", "105-59-9": "MDEA", "124-38-9": "CO2", "110-85-0": "PZ"}
cols = ["cas", "name", "n", "n_low", "n_high", "scheme", "dipole_scaling", "polarizability_scaling", "correlation_integral_parameter",
        "MAD", "MARD%", "RMSE", "max_RD%", "T_min", "T_max", "rueben_scheme", "status"]
r[r.cas.isin(target)][cols].round(4)